# Module 05 — Shuffle & Sort

Join strategies, Storage Partitioned Join (SPJ), and table ordering.

In [ ]:
import sys
sys.path.insert(0, '/workspace' if __import__('pathlib').Path('/workspace').exists() else '.')
from shared.spark_session import get_spark
from shared import config

spark = get_spark('module-05-shuffle-sort')
trips = spark.read.parquet(*config.yellow_taxi_paths())
zones = spark.read.option('header', True).csv(config.zones_path())

## 1. Broadcast Join

In [ ]:
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '50MB')
small = zones.limit(50)
plan = trips.join(small.withColumnRenamed('LocationID', 'PULocationID'), 'PULocationID')
plan.explain('formatted')
# Look for BroadcastHashJoin

## 2. Sort-Merge Join (disable broadcast)

In [ ]:
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')
plan = trips.join(zones.withColumnRenamed('LocationID', 'PULocationID'), 'PULocationID')
plan.explain('formatted')
# Look for SortMergeJoin + Exchange

## 3. Storage Partitioned Join (SPJ)

When both Iceberg tables share compatible partitioning, Spark can avoid shuffle.

In [ ]:
spark.sql('''
    SET spark.sql.sources.v2.bucketing.enabled=true;
    SET spark.sql.sources.v2.bucketing.pushPartValues.enabled=true;
    SET spark.sql.iceberg.planning.preserve-data-grouping=true;
    SET spark.sql.requireAllClusterKeysForCoPartition=false;
    SET spark.sql.autoBroadcastJoinThreshold=-1;
''')

In [ ]:
trips_a = trips.select('VendorID', 'PULocationID', 'total_amount', 'tpep_pickup_datetime')
trips_b = trips.select('VendorID', 'DOLocationID', 'total_amount', 'tpep_dropoff_datetime')

# Write two Iceberg tables partitioned identically
trips_a.writeTo('local.default.trips_a').using('iceberg').partitionedBy('PULocationID').createOrReplace()
trips_b.writeTo('local.default.trips_b').using('iceberg').partitionedBy('PULocationID').createOrReplace()

In [ ]:
spj = spark.sql('''
    SELECT a.VendorID, a.total_amount, b.total_amount
    FROM local.default.trips_a a
    JOIN local.default.trips_b b
    ON a.PULocationID = b.PULocationID AND a.VendorID = b.VendorID
''')
spj.explain('formatted')
# With SPJ: no Exchange before join
spj.limit(10).show()

## 4. Table-Level Ordering

`WRITE ORDERED BY` sets `write.distribution-mode=range`, causing sort on write.

In [ ]:
spark.sql('''
    CREATE OR REPLACE TABLE local.default.trips_ordered
    USING iceberg
    AS SELECT * FROM local.default.trips_a
''')
spark.sql('ALTER TABLE local.default.trips_ordered WRITE ORDERED BY (PULocationID, total_amount)')
spark.sql('SHOW TBLPROPERTIES local.default.trips_ordered').filter("key LIKE 'write.%' OR key = 'sort-order'").show(truncate=False)

In [ ]:
# Write with pre-sorted data, skip table ordering enforcement
sorted_df = trips_a.orderBy('PULocationID', 'total_amount')
sorted_df.writeTo('local.default.trips_presorted').using('iceberg') \
    .option('use-table-distribution-and-ordering', 'false') \
    .createOrReplace()

## Key Takeaways

1. Broadcast join avoids shuffle for small tables.
2. SPJ eliminates shuffle when Iceberg partition specs align.
3. Table sort order improves read performance but adds write cost.